In [ ]:
import os
import tensorflow as tf
from tensorflow.keras import layers, losses, Sequential, callbacks
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import cifar10

In [ ]:
train_ds, test_ds = cifar10.load_data()

split = int(train_ds[0].shape[0]*.8)

train_ds, val_ds = (
train_ds[0][:split], train_ds[1][:split]), (train_ds[0][split:], train_ds[1][split:])


aug = tf.keras.Sequential([
    layers.Rescaling(1/255),
    layers.RandomFlip(mode="horizontal"),
    layers.RandomRotation(factor=.1),
    layers.RandomContrast(factor=(.2,.9)),
    layers.RandomZoom(height_factor=(.1,.3))
])

def preprocess(x, y):
  return aug(x), tf.one_hot(tf.squeeze(y), 10)

train_ds = tf.data.Dataset.from_tensor_slices(train_ds).map(preprocess).batch(64)
val_ds = tf.data.Dataset.from_tensor_slices(val_ds).map(preprocess).batch(64)
test_ds = tf.data.Dataset.from_tensor_slices(test_ds).map(preprocess).batch(64)

In [ ]:
train_ds

In [ ]:
base_model = tf.keras.applications.VGG19(
    include_top=False, weights="imagenet", input_shape=(32,32,3))
base_model.summary()

In [ ]:
for layer in base_model.layers[:-5]:
  layer.trainable=False
base_model.summary()

In [ ]:
model = Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256),
    layers.ReLU(),
    layers.Dense(128),
    layers.ReLU(),
    layers.Dense(64),
    layers.ReLU(),
    layers.Dense(10),
    layers.Softmax()
])
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss=losses.CategoricalCrossentropy(),
    metrics=["accuracy"]
)

In [ ]:
model.fit(
  train_ds,
  validation_data = val_ds,
  epochs=1,
  callbacks =[
      callbacks.TensorBoard(log_dir="logs/ex6")
  ]
)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/ex6

In [ ]:
model.evaluate(test_ds)

In [ ]:
image, label = next(iter(val_ds))
images = image.numpy()[:9]

res = tf.argmax(model(image), axis=-1)

In [ ]:
plt.suptitle("Inference")
for i, (img, r) in enumerate(zip(images, res)):
    plt.subplot(331+i)
    plt.title( f"{tf.argmax(label[i])} - {res[i]}")
    plt.imshow(img)
    plt.axis("off")
plt.show()